<a href="https://colab.research.google.com/github/f1725developmenttechnologies-create/klarixa-ecosistema-ia/blob/main/Klarixa_ecosistema_ia_%F0%9F%A7%A0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# KLARIXA IA - NOTEBOOK 3: MESH ORCHESTRATOR HUB
# ==========================================

import os, subprocess, sys, asyncio

packages = ["fastapi", "uvicorn", "pyngrok", "pydantic", "httpx", "nest_asyncio"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

import uvicorn, nest_asyncio, httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

NGROK_AUTHTOKEN = "3BGz0Rj9E7t698fooa8Abz30dKn_5gmd7DXoLjDxgoFZN74pW"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Enlace a las URLs activas de los Nodos 1 y 2
BRAIN1_URL = "https://constance-spoilless-anderson.ngrok-free.dev"
BRAIN2_URL = "https://constance-spoilless-anderson.ngrok-free.dev"

app = FastAPI(title="KLARIXA Master Orchestrator Node")

class MasterQuery(BaseModel):
    prompt: str
    target_brain: str = "auto"

@app.get("/health")
async def health_mesh():
    status = {"orchestrator": "online", "brain1": "unknown", "brain2": "unknown"}
    async with httpx.AsyncClient() as client:
        try:
            r1 = await client.get(f"{BRAIN1_URL}/health", timeout=3.0)
            status["brain1"] = "connected" if r1.status_code == 200 else "degraded"
        except:
            status["brain1"] = "offline"

        try:
            r2 = await client.get(f"{BRAIN2_URL}/health", timeout=3.0)
            status["brain2"] = "connected" if r2.status_code == 200 else "degraded"
        except:
            status["brain2"] = "offline"

    return status

@app.post("/api/chat")
async def route_request(query: MasterQuery):
    async with httpx.AsyncClient() as client:
        selected_brain = query.target_brain
        if selected_brain == "auto":
            keywords_code = ["def ", "code", "python", "script", "function", "api", "json"]
            selected_brain = "brain2" if any(k in query.prompt.lower() for k in keywords_code) else "brain1"

        target_url = BRAIN1_URL if selected_brain == "brain1" else BRAIN2_URL
        fallback_url = BRAIN2_URL if selected_brain == "brain1" else BRAIN1_URL

        try:
            res = await client.post(f"{target_url}/process", json={"prompt": query.prompt}, timeout=30.0)
            return res.json()
        except Exception:
            res = await client.post(f"{fallback_url}/process", json={"prompt": query.prompt}, timeout=30.0)
            return res.json()

port = 8000
public_url = ngrok.connect(port)

print("\n" + "="*60)
print(f"👑 KLARIXA MASTER ORCHESTRATOR ACTIVE: {public_url.public_url}")
print(f"🔗 URL Final para Base44 / Termux: {public_url.public_url}/api/chat")
print(f"📊 Estado Malla: {public_url.public_url}/health")
print("="*60 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=port, loop="asyncio")
server = uvicorn.Server(config)
asyncio.create_task(server.serve())


👑 KLARIXA MASTER ORCHESTRATOR ACTIVE: https://constance-spoilless-anderson.ngrok-free.dev
🔗 URL Final para Base44 / Termux: https://constance-spoilless-anderson.ngrok-free.dev/api/chat
📊 Estado Malla: https://constance-spoilless-anderson.ngrok-free.dev/health



<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.13/dist-packages/uvicorn/server.py:79>>

In [ ]:
import os
import httpx
import asyncio
from pydantic import BaseModel

# 1. Configuración de RPCs para Base Network y endpoints de ingesta
BASE_MAINNET_RPC = os.getenv("BASE_RPC_URL", "https://mainnet.base.org")
BASE_SEPOLIA_RPC = os.getenv("BASE_SEPOLIA_RPC_URL", "https://sepolia.base.org")

# 2. Cerebro 3: Crawler & Network Data Ingestor
class BaseNetworkCrawler:
    def __init__(self, rpc_url: str):
        self.rpc_url = rpc_url
        self.client = httpx.AsyncClient(timeout=10.0)

    async def fetch_network_state(self) -> dict:
        payload = {
            "jsonrpc": "2.0",
            "method": "eth_blockNumber",
            "params": [],
            "id": 1
        }
        try:
            resp = await self.client.post(self.rpc_url, json=payload)
            block_hex = resp.json().get("result", "0x0")
            block_num = int(block_hex, 16)
            return {
                "module": "CEREBRO_3_CRAWLER",
                "network": "Base Mainnet",
                "latest_block": block_num,
                "status": "ONLINE"
            }
        except Exception as e:
            return {
                "module": "CEREBRO_3_CRAWLER",
                "network": "Base Mainnet",
                "status": "OFFLINE",
                "error": str(e)
            }

# 3. Ejecución del Módulo 3
async def run_cerebro_3():
    print("📡 [Cerebro 3] Inicializando Crawler e ingesta de datos de la red Base...")
    crawler = BaseNetworkCrawler(BASE_MAINNET_RPC)
    state = await crawler.fetch_network_state()

    print("\n✅ ESTADO DEL CEREBRO 3 (INGESTA):")
    print(state)

await run_cerebro_3()

📡 [Cerebro 3] Inicializando Crawler e ingesta de datos de la red Base...

✅ ESTADO DEL CEREBRO 3 (INGESTA):
{'module': 'CEREBRO_3_CRAWLER', 'network': 'Base Mainnet', 'latest_block': 50703363, 'status': 'ONLINE'}
